# DeepGuard AI - Notebook 02: Video Preprocessing & Face Extraction
### Project: Deepfake Image & Video Detection System
**Objective:** Controlled frame extraction from raw video sequences and face cropping with bounding box margin expansion.
Generates the preprocessed face crops dataset for fast training.

### Step 1: Setup & Dependencies
Install face detection and video processing libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install opencv-python-headless pillow tqdm pandas
import cv2
import os
import glob
import pandas as pd
from PIL import Image
from tqdm import tqdm

### Step 2: Face Extractor with Margin Expansion
Forensic analysis requires capturing forehead, jawline, and ear-boundary blending seams.
We apply a 25% margin expansion around the detected face bounding box.

In [ ]:
class ColabFaceExtractor:
    def __init__(self, margin=0.25):
        self.margin = margin
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.cascade = cv2.CascadeClassifier(cascade_path)

    def extract_face(self, frame_rgb, target_size=(224, 224)):
        gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
        h_img, w_img = frame_rgb.shape[:2]
        faces = self.cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

        if len(faces) == 0:
            return None

        # Select largest face by area
        x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
        dx = int(w * self.margin)
        dy = int(h * self.margin)

        x1 = max(0, x - dx)
        y1 = max(0, y - dy)
        x2 = min(w_img, x + w + dx)
        y2 = min(h_img, y + h + dy)

        crop = frame_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        return cv2.resize(crop, target_size, interpolation=cv2.INTER_LINEAR)

### Step 3: Controlled Frame Sampling
Sample frames at fixed intervals (e.g., 1 frame every 0.5s or every 15 frames) up to 30 frames per video.

In [ ]:
def process_video_to_face_crops(video_path, output_dir, extractor, max_frames=30, frame_step=15):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    saved_paths = []
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_idx = 0
    saved_count = 0

    while True:
        ret, frame_bgr = cap.read()
        if not ret or saved_count >= max_frames:
            break

        if frame_idx % frame_step == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            face_crop = extractor.extract_face(frame_rgb)
            if face_crop is not None:
                out_path = os.path.join(output_dir, f"{base_name}_frame{frame_idx:04d}.jpg")
                Image.fromarray(face_crop).save(out_path, quality=95)
                saved_paths.append(out_path)
                saved_count += 1

        frame_idx += 1

    cap.release()
    return saved_paths

### Step 4: Batch Preprocessing Execution
Run across your train, validation, and test splits to produce clean CSV manifests.

In [ ]:
# Configure directories
OUTPUT_DIR = "/content/deepfake_face_crops"
os.makedirs(os.path.join(OUTPUT_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "val"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "test"), exist_ok=True)

print("Preprocessed face crop pipeline ready. Proceed to run extraction on your dataset.")